# LangGraph Persistent Checkpointers: SQLite and MongoDB


In [1]:
import os
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model

load_dotenv()

llm = init_chat_model("groq:qwen/qwen3-32b")

/Users/krishrohilla/Desktop/MindzKonnected/Tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

## 1. SQLite 

In [3]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "chatbot_memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)

sqlite_checkpointer = SqliteSaver(conn)

sqlite_graph = builder.compile(checkpointer=sqlite_checkpointer)
print("SQLite graph compiled successfully!")

SQLite graph compiled successfully!


In [4]:
config = {"configurable": {"thread_id": "sqlite_thread_1"}}

response = sqlite_graph.invoke({"messages": "Hi, my name is Krish and I like cricket!"}, config=config)
print("Assistant Response 1:")
print(response["messages"][-1].content)

Assistant Response 1:
<think>
Okay, the user just repeated their introduction again: "Hi, my name is Krish and I like cricket!" Let me check the conversation history.

In the first message, Krish introduced themselves, and I responded by asking about teams or players. Then Krish asked, "What is my name and what sport do I like?" and I confirmed their name and sport again. Now they're repeating the same initial message. 

Hmm, maybe they're testing if the AI remembers previous interactions. Since the conversation history is there, I should acknowledge that they've already mentioned their name and sport. But I should also keep it friendly and open for more discussion. 

I need to make sure not to just repeat the same response. Perhaps add something new, like mentioning a recent cricket event or asking a follow-up question to keep the dialogue engaging. Let me check the latest cricket news quickly. Oh, the ICC Men's T20 World Cup is coming up in 2024. That's a good topic to bring in.

So,

In [5]:
# Send second message to verify
response = sqlite_graph.invoke({"messages": "What is my name and what sport do I like?"}, config=config)
print("Assistant Response 2:")
print(response["messages"][-1].content)

Assistant Response 2:
<think>
Okay, so the user is named Krish and he likes cricket. Let me look back at the conversation history to make sure I get this right.

First, Krish introduced himself and mentioned liking cricket. I responded by acknowledging his name and interest, and asked about teams or players. Then he asked again about his name and sport, so I confirmed it again. Now, he's repeating the same question. 

Hmm, maybe he wants to test if I remember or if I'm paying attention. Alternatively, he might be practicing for something or trying to see how persistent the chatbot is with remembering details. I should make sure to confirm his name and sport again, but also add something new to keep the conversation going. 

In the last interaction, I mentioned the ICC Men's T20 World Cup 2024. Maybe he's interested in that. I should bring that up again to show I'm keeping track of previous topics. Also, since he likes cricket, asking about his favorite teams or players could be a good 

## 2. MongoDB 

In [ ]:
import mongomock
from mongomock.collection import BulkOperationBuilder
original_add_update = BulkOperationBuilder.add_update

def patched_add_update(self, *args, **kwargs):
    kwargs.pop('sort', None)
    return original_add_update(self, *args, **kwargs)

BulkOperationBuilder.add_update = patched_add_update

from langgraph.checkpoint.mongodb import MongoDBSaver

# For production, use standard client:
# from pymongo import MongoClient
# client = MongoClient("mongodb://localhost:27017")

# Simulated MongoDB Client for demonstration
client = mongomock.MongoClient()

mongo_checkpointer = MongoDBSaver(client)

mongo_graph = builder.compile(checkpointer=mongo_checkpointer)
print("MongoDB graph compiled successfully!")

MongoDB graph compiled successfully!


In [7]:
config = {"configurable": {"thread_id": "mongo_thread_1"}}

response = mongo_graph.invoke({"messages": "Hello! My favorite color is green."}, config=config)
print("Assistant Response 1:")
print(response["messages"][-1].content)

Assistant Response 1:
<think>
Okay, the user mentioned their favorite color is green. I should respond positively and show enthusiasm. Maybe ask a follow-up question to keep the conversation going. I can mention something about green, like nature or symbols associated with it. Let me make sure to keep it friendly and open-ended. Also, avoid any assumptions beyond what they've shared. Maybe suggest a related topic or activity? Hmm, something like visiting a park or discussing why they like green. Yeah, that sounds good. Keep the tone warm and inviting.
</think>

Hello! 🌿 Green is such a wonderful choice—so refreshing and connected to nature! I'm curious, does your love for green come from the vibrant forests, the calming feel of spring, or maybe something else? I'd love to hear what makes green special to you! 🌳💚


In [8]:
# Send second message to verify 
response = mongo_graph.invoke({"messages": "Do you remember what my favorite color is?"}, config=config)
print("Assistant Response 2:")
print(response["messages"][-1].content)

Assistant Response 2:
<think>
Okay, the user is asking if I remember their favorite color. Let me check the conversation history. They mentioned earlier that their favorite color is green, and in my previous response, I acknowledged it and asked about their connection to green. Now they’re confirming if I remember.

I need to make sure I clearly state that I do remember. Maybe start with a positive affirmation like "Of course!" to show I'm paying attention. Then, I can reinforce the memory by referencing the color's significance, like nature or growth. Adding an emoji related to green would be nice for visual confirmation. Also, since they asked why I asked before, maybe they want to continue the conversation. I should invite them to share more or ask if there's something specific they want to talk about related to green. That keeps the interaction engaging and shows I'm interested in their thoughts. Let me structure the response to be friendly and encouraging.
</think>

Of course! 🌿 Y

In [9]:

conn.close()
print("SQLite connection closed.")


SQLite connection closed.
